<a href="https://colab.research.google.com/github/genaiconference/AV-DHS-2026/blob/main/kgbuilder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movies Knowledge Graph
## Custom `neo4j-graphrag` Pipeline

This notebook builds a movie knowledge graph from the **TMDB + IMDb Merged Movies Dataset** using the **fully-componentised `neo4j_graphrag` `Pipeline`** (instead of `SimpleKGPipeline`).

## Pipeline Structure
1. **Data loader** – custom loader that reads the movies CSV and emits a `PdfDocument` wrapping a list of per-movie dicts.
2. **Text splitter** – custom splitter that turns each movie row into a `TextChunk` with metadata (title, release_date, dataset).
3. **Chunk embedder** – embeds each chunk text with OpenAI embeddings.
4. **Schema builder** – manual `NodeType` / `RelationshipType` / `patterns` for the movie domain (Movie, Person, Genre, ProductionCompany, Country, Language, Keyword, CastMember).
5. **Entity & relation extractor** – `LLMEntityRelationExtractor` grounded by the schema.
6. **KG writer** – `Neo4jWriter` persists the extracted graph to Aura.
7. **Entity resolver** – `SinglePropertyExactMatchResolver` (run twice: once on `title`, once on `name`).
8. **Entity embedder** (custom `Component`) – embeds the longest string property of each entity.
9. **Pipeline assembly** – `Pipeline().add_component(...)` + `pipe.connect(...)` + `await pipe.run(...)`.
10. **Validation Cypher queries** – sanity checks against the new movie schema (`CAST_IN`, `DIRECTED_BY`, `HAS_GENRE`, …).

> Each section can be run **independently** for debugging, then re-run end-to-end with the assembled pipeline at the bottom.

## 0. Install required packages

In [ ]:
# Run once per environment. Comment out after the first successful install.
#%pip install --quiet neo4j-graphrag[openai] neo4j openai pandas python-dotenv tqdm

## 1. Credentials & Drivers

We use the same Neo4j Free Aura instance and OpenAI key.

In [ ]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv(dotenv_path='C:\\Users\\VURIMKR1\\Documents\\AV2026\\.env')
except Exception:
    print("error reading env details")
    pass

# --- Neo4j Aura ---
NEO4J_URI      = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE')

# --- OpenAI ---
os.environ.setdefault(
    'OPENAI_API_KEY',
    os.getenv('OPENAI_API_KEY')
)

print('NEO4J_URI :', NEO4J_URI)
print('OPENAI key set:', bool(os.environ.get('OPENAI_API_KEY')))

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print('Connected to Neo4j ✔')

## 2. LLM & Embedder

Same approach as the reference notebook: two `OpenAILLM` instances (one for extraction, one for strict JSON outputs) plus an `OpenAIEmbeddings` instance.

In [ ]:
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings

# Main LLM used by LLMEntityRelationExtractor
llm = OpenAILLM(
    model_name='gpt-5-mini',
    #model_params={'temperature': 0},
)

# Strict-JSON LLM (handy for schema extraction or community summarisation)
json_llm = OpenAILLM(
    model_name='gpt-5-mini',
    model_params={
        #'temperature': 0,
        'response_format': {'type': 'json_object'},
    },
)

embedder = OpenAIEmbeddings(model='text-embedding-3-small')
print('LLM + embedder ready ✔')

## 3. Custom Data Loader

Reads the TMDB+IMDb CSV, filters to movies released after the year 2000, and returns a single `PdfDocument` whose `text` is a string representation of a list of per-movie dicts.

This mirrors the reference notebook's `PickleDataLoader` pattern — the loader output is opaque to the rest of the pipeline; the custom splitter knows how to decode it.

In [ ]:
from pathlib import Path
import pandas as pd
import math
from neo4j_graphrag.experimental.components.pdf_loader import DataLoader, PdfDocument

csv_path = os.getnev('MOVIES_CSV_PATH')

DATA_PATH = Path(csv_path)

def _safe_str(x) -> str:
    """Coerce any value (incl. NaN) into a clean string."""
    if x is None:
        return ''
    if isinstance(x, float) and math.isnan(x):
        return ''
    return str(x).strip()


class MoviesCSVDataLoader(DataLoader):
    """Custom DataLoader that reads a movies CSV and returns a PdfDocument.

    The `text` field of the returned PdfDocument is a `repr()` of a list of
    per-movie dicts. The splitter (next step) will `ast.literal_eval` it.
    """

    def __init__(self, year_min: int = 2000, limit: int | None = 40):
        self.year_min = year_min
        self.limit = limit

    async def run(self, path: Path) -> PdfDocument:
        df = pd.read_csv(path)

        # Filter to movies released after `year_min`
        if 'release_date' in df.columns:
            df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
            df = df[df['release_date'].dt.year > self.year_min]
        df = df.reset_index(drop=True)

        if self.limit is not None:
            df = df.head(self.limit)

        # Build a list of normalised per-movie dicts
        records = []
        for _, row in df.iterrows():
            records.append({
                'title':                _safe_str(row.get('title')),
                'release_date':         _safe_str(row.get('release_date')),
                'vote_average':         _safe_str(row.get('vote_average')),
                'revenue':              _safe_str(row.get('revenue')),
                'runtime':              _safe_str(row.get('runtime')),
                'budget':               _safe_str(row.get('budget')),
                'overview':             _safe_str(row.get('overview')),
                'tagline':              _safe_str(row.get('tagline')),
                'genres':               _safe_str(row.get('genres')),
                'production_companies': _safe_str(row.get('production_companies')),
                'production_countries': _safe_str(row.get('production_countries')),
                'spoken_languages':     _safe_str(row.get('spoken_languages')),
                'keywords':             _safe_str(row.get('keywords')),
                'directors':            _safe_str(row.get('directors')),
                'cast':                 _safe_str(row.get('cast')),
            })

        print(f'Loaded {len(records)} movie rows from {path.name} (year > {self.year_min}).')

        return PdfDocument(
            text=repr(records),
            document_info={'path': str(path)},
        )

# Smoke test
loader = MoviesCSVDataLoader(year_min=2000, limit=40)
page_data = await loader.run(DATA_PATH)
print('PdfDocument text length:', len(page_data.text))

## 4. Custom Text Splitter

Decodes the `PdfDocument` produced by the loader and turns each movie record into a `TextChunk` with rich metadata. The chunk `text` is a human/LLM-friendly multi-line summary of the movie (mirroring the `row_to_movie_doc(...)` helper used in `01_first_nb.ipynb`).

In [ ]:
import ast
from neo4j_graphrag.experimental.components.text_splitters.base import TextSplitter
from neo4j_graphrag.experimental.components.types import TextChunks, TextChunk, PdfDocument

def _row_to_movie_doc(rec: dict) -> str:
    """Build a multi-line, LLM-friendly text block for a single movie row."""
    lines = []
    lines.append(f"Title: {rec.get('title','')}")
    if rec.get('release_date'):  lines.append(f"Release date: {rec['release_date']}")
    if rec.get('runtime'):       lines.append(f"Runtime (minutes): {rec['runtime']}")
    if rec.get('budget'):        lines.append(f"Budget (USD): {rec['budget']}")
    if rec.get('revenue'):       lines.append(f"Revenue (USD): {rec['revenue']}")
    if rec.get('vote_average'):  lines.append(f"Vote average: {rec['vote_average']}")
    if rec.get('genres'):                lines.append(f"Genres: {rec['genres']}")
    if rec.get('production_companies'):  lines.append(f"Production companies: {rec['production_companies']}")
    if rec.get('production_countries'):  lines.append(f"Production countries: {rec['production_countries']}")
    if rec.get('spoken_languages'):      lines.append(f"Spoken languages: {rec['spoken_languages']}")
    if rec.get('keywords'):              lines.append(f"Keywords: {rec['keywords']}")
    if rec.get('directors'):             lines.append(f"Director(s): {rec['directors']}")
    if rec.get('cast'):                  lines.append(f"Cast: {rec['cast']}")
    if rec.get('tagline'):               lines.append(f"Tagline: {rec['tagline']}")
    if rec.get('overview'):              lines.append(f"Overview: {rec['overview']}")
    return '\n'.join(lines)


class MoviesRowTextSplitter(TextSplitter):
    """Split the loader's PdfDocument into one TextChunk per movie row."""

    def __init__(self, dataset_name: str = 'TMDB+IMDb Movies'):
        self.dataset_name = dataset_name

    async def run(self, page_data: PdfDocument) -> TextChunks:
        raw = page_data['text'] if isinstance(page_data, dict) else page_data.text
        records = ast.literal_eval(raw)

        chunks = []
        for i, rec in enumerate(records):
            chunks.append(TextChunk(
                index=i,
                text=_row_to_movie_doc(rec),
                metadata={
                    'title':        rec.get('title', ''),
                    'release_date': rec.get('release_date', ''),
                    'dataset':      self.dataset_name,
                },
            ))
        print(f'Built {len(chunks)} TextChunks (one per movie).')
        return TextChunks(chunks=chunks)


# Smoke test
splitter = MoviesRowTextSplitter()
text_chunks = await splitter.run(page_data)
print('First chunk preview:\n', text_chunks.chunks[0].text[:500])

## 5. Chunk Embedder

Adds an embedding to every chunk so we can later run vector search against the movie documents.

In [ ]:
from neo4j_graphrag.experimental.components.embedder import TextChunkEmbedder

text_chunk_embedder = TextChunkEmbedder(embedder=embedder)

# Smoke test (component-level)
embedded_chunks = await text_chunk_embedder.run(text_chunks=text_chunks)
print('Embedded chunks:', len(embedded_chunks.chunks))
print('First chunk embedding length:', len(embedded_chunks.chunks[0].metadata.get('embedding', [])))

## 6. Schema (manual)

We reuse the upgraded movie schema from `01_first_nb.ipynb`:

**Nodes**: Movie · Person · Genre · ProductionCompany · Country · Language · Keyword · CastMember  
**Relationships**: HAS_GENRE · PRODUCED_BY · PRODUCED_IN · SPOKEN_IN · TAGGED_WITH · DIRECTED_BY · WRITTEN_BY · CAST_IN

Patterns describe `(source) -[REL]-> (target)` triplets and are used both by the extractor (to constrain the LLM) and by the graph pruner (to discard out-of-schema edges).

In [ ]:
from neo4j_graphrag.experimental.components.schema import (
    SchemaBuilder,
    NodeType,
    RelationshipType,
    GraphSchema,
)

schema_builder = SchemaBuilder()

# -------- Node Types --------
node_types = [
    NodeType(
        label='Movie',
        description='A motion picture / film with metadata such as title, runtime, budget, revenue, release date and ratings.',
        properties=[
            {'name': 'title',         'type': 'STRING',  'description': 'Title of the movie.'},
            {'name': 'release_date',  'type': 'DATE',    'description': 'Theatrical release date (YYYY-MM-DD).'},
            {'name': 'runtime',       'type': 'INTEGER', 'description': 'Runtime in minutes.'},
            {'name': 'budget',        'type': 'INTEGER', 'description': 'Production budget in USD.'},
            {'name': 'revenue',       'type': 'INTEGER', 'description': 'Worldwide box-office revenue in USD.'},
            {'name': 'vote_average',  'type': 'FLOAT',   'description': 'Average user rating (typically 0–10).'},
            {'name': 'overview',      'type': 'STRING',  'description': 'Plot synopsis.'},
            {'name': 'tagline',       'type': 'STRING',  'description': 'Marketing tagline of the movie.'},
        ],
        additional_properties=True,
    ),
    NodeType(
        label='Person',
        description='A real person involved in a movie as director, writer, actor or other crew.',
        properties=[
            {'name': 'name', 'type': 'STRING', 'description': 'Full name of the person.'},
        ],
        additional_properties=True,
    ),
    NodeType(
        label='Genre',
        description='A film genre such as Action, Drama, Comedy, Horror, Science Fiction.',
        properties=[
            {'name': 'name', 'type': 'STRING', 'description': 'Canonical genre name.'},
        ],
        additional_properties=True,
    ),
    NodeType(
        label='ProductionCompany',
        description='A company that produced or financed the movie.',
        properties=[
            {'name': 'name', 'type': 'STRING', 'description': 'Name of the production company.'},
        ],
        additional_properties=True,
    ),
    NodeType(
        label='Country',
        description='A country where the movie was produced.',
        properties=[
            {'name': 'name', 'type': 'STRING', 'description': 'Country name.'},
        ],
        additional_properties=True,
    ),
    NodeType(
        label='Language',
        description='A spoken language in the movie.',
        properties=[
            {'name': 'name', 'type': 'STRING', 'description': 'Language name (e.g. English, French).'},
        ],
        additional_properties=True,
    ),
    NodeType(
        label='Keyword',
        description='A descriptive tag / theme attached to the movie (e.g. "time travel", "heist").',
        properties=[
            {'name': 'name', 'type': 'STRING', 'description': 'Keyword text.'},
        ],
        additional_properties=True,
    ),
    NodeType(
        label='CastMember',
        description='A character portrayed in the movie (distinct from the real Person who plays it).',
        properties=[
            {'name': 'name', 'type': 'STRING', 'description': 'Character name.'},
        ],
        additional_properties=True,
    ),
]

# -------- Relationship Types --------
relationship_types = [
    RelationshipType(label='HAS_GENRE',  description='Movie -[HAS_GENRE]-> Genre',                 properties=[], additional_properties=True),
    RelationshipType(label='PRODUCED_BY',description='Movie -[PRODUCED_BY]-> ProductionCompany',   properties=[], additional_properties=True),
    RelationshipType(label='PRODUCED_IN',description='Movie -[PRODUCED_IN]-> Country',             properties=[], additional_properties=True),
    RelationshipType(label='SPOKEN_IN',  description='Movie -[SPOKEN_IN]-> Language',              properties=[], additional_properties=True),
    RelationshipType(label='TAGGED_WITH',description='Movie -[TAGGED_WITH]-> Keyword',             properties=[], additional_properties=True),
    RelationshipType(label='DIRECTED_BY',description='Movie -[DIRECTED_BY]-> Person (director)',   properties=[], additional_properties=True),
    RelationshipType(label='WRITTEN_BY', description='Movie -[WRITTEN_BY]-> Person (writer)',      properties=[], additional_properties=True),
    RelationshipType(label='CAST_IN',    description='Person -[CAST_IN]-> Movie (actor appears in movie)', properties=[], additional_properties=True),
]

# -------- Patterns (allowed triplets) --------
patterns = [
    ('Movie',  'HAS_GENRE',   'Genre'),
    ('Movie',  'PRODUCED_BY', 'ProductionCompany'),
    ('Movie',  'PRODUCED_IN', 'Country'),
    ('Movie',  'SPOKEN_IN',   'Language'),
    ('Movie',  'TAGGED_WITH', 'Keyword'),
    ('Movie',  'DIRECTED_BY', 'Person'),
    ('Movie',  'WRITTEN_BY',  'Person'),
    ('Person', 'CAST_IN',     'Movie'),
]

manual_schema = GraphSchema(
    node_types=node_types,
    relationship_types=relationship_types,
    patterns=patterns,
)

print('Schema ready:')
print('  node_types        :', [n.label for n in node_types])
print('  relationship_types:', [r.label for r in relationship_types])
print('  patterns          :', len(patterns))

## 7. Entity & Relation Extractor

Uses `LLMEntityRelationExtractor`. The schema we built above is passed in via the pipeline; here we just configure the extractor itself.

In [ ]:
from neo4j_graphrag.experimental.components.entity_relation_extractor import (
    LLMEntityRelationExtractor,
    OnError,
)

entity_extractor = LLMEntityRelationExtractor(
    llm=llm,
    on_error=OnError.IGNORE,        # don't blow up the pipeline on a single bad chunk
    create_lexical_graph=True,      # also create Document/Chunk nodes and FROM_CHUNK edges
)
print('LLMEntityRelationExtractor ready ✔')

## 8. KG Writer (Neo4jWriter)

Persists the extracted graph into Aura.

In [ ]:
from neo4j_graphrag.experimental.components.kg_writer import Neo4jWriter

eg_writer = Neo4jWriter(
    driver=driver,
    neo4j_database=NEO4J_DATABASE,
    batch_size=1000,
)
print('Neo4jWriter ready ✔')

## 9. Entity Resolver

We use exact-match resolution on the most discriminating string property of each node label:
* `Movie` nodes share `title`
* All other `__Entity__` nodes (Person, Genre, …) share `name`

We run the resolver twice — once on `title`, once on `name` — so that both axes are deduplicated.

In [ ]:
from neo4j_graphrag.experimental.components.resolver import SinglePropertyExactMatchResolver

resolver_title = SinglePropertyExactMatchResolver(driver=driver, resolve_property='title')
resolver_name  = SinglePropertyExactMatchResolver(driver=driver, resolve_property='name')
print('Resolvers ready ✔')

## 10. Custom Component — Entity Text Embedder

For every `__Entity__` node, find the **longest string property** and embed it. The embedding is stored on the node as `e.embedding`, enabling later vector search at the entity level (in addition to chunk-level vectors created by step 5).

This is the **exact same pattern** as `EntityTextEmbedder` from the reference notebook.

In [ ]:
from neo4j_graphrag.experimental.pipeline import Component, DataModel

class EmbedEntityResult(DataModel):
    updated_count: int


class EntityTextEmbedder(Component):
    """For every `:__Entity__` node, embed its longest string property → e.embedding."""

    def __init__(self, driver, label: str = '__Entity__', embedder=None):
        self.driver = driver
        self.label = label
        self.embedder = embedder

    async def run(self) -> EmbedEntityResult:
        updated = 0
        with self.driver.session(database=NEO4J_DATABASE) as session:
            result = session.run(
                f'MATCH (e:`{self.label}`) RETURN elementId(e) AS id, properties(e) AS props'
            )
            for record in result:
                node_id = record['id']
                props = record['props'] or {}
                str_props = {k: v for k, v in props.items() if isinstance(v, str) and v.strip()}
                if not str_props:
                    continue
                _, text = max(str_props.items(), key=lambda kv: len(kv[1]))
                emb = self.embedder.embed_query(text)
                session.run(
                    'MATCH (e) WHERE elementId(e) = $id SET e.embedding = $embedding',
                    id=node_id, embedding=emb,
                )
                updated += 1
        print(f'Embedded {updated} entity nodes.')
        return EmbedEntityResult(updated_count=updated)


entity_embedder = EntityTextEmbedder(driver=driver, embedder=embedder)
print('EntityTextEmbedder ready ✔')

## 11. Wipe target graph (optional but recommended for clean re-runs)

In [ ]:
WIPE_BEFORE_RUN = True

if WIPE_BEFORE_RUN:
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run('MATCH (n) DETACH DELETE n')
    print('Graph wiped ✔')
else:
    print('Skipping wipe (WIPE_BEFORE_RUN = False).')

## 12. Assemble & run the custom Pipeline

This is the heart of the notebook: components are wired explicitly with `pipe.connect(...)`. The data flow is:

```
data_loader → text_splitter → chunk_embedder ─┐
                            schema_builder ───┤→ entity_extractor → eg_writer ┬→ resolver_title
                                                                              ├→ resolver_name
                                                                              └→ entity_embedder
```

The `input_config` mapping on each `connect()` call tells the downstream component which upstream output to consume.

In [ ]:
from neo4j_graphrag.experimental.pipeline import Pipeline

pipe = Pipeline()

# 1. Register components
pipe.add_component(loader,              'data_loader')
pipe.add_component(splitter,            'text_splitter')
pipe.add_component(text_chunk_embedder, 'chunk_embedder')
pipe.add_component(schema_builder,      'schema')
pipe.add_component(entity_extractor,    'entity_extractor')
pipe.add_component(eg_writer,           'eg_writer')
pipe.add_component(resolver_title,      'resolver_title')
pipe.add_component(resolver_name,       'resolver_name')
pipe.add_component(entity_embedder,     'entity_embedder')

# 2. Wire data flow
pipe.connect('data_loader',    'text_splitter',    input_config={'page_data':   'data_loader'})
pipe.connect('text_splitter',  'chunk_embedder',   input_config={'text_chunks': 'text_splitter'})
pipe.connect('chunk_embedder', 'entity_extractor', input_config={'chunks':      'chunk_embedder'})
pipe.connect('schema',         'entity_extractor', input_config={'schema':      'schema'})
pipe.connect('entity_extractor','eg_writer',       input_config={'graph':       'entity_extractor'})

# Post-write components — they operate directly on the DB, so no input mapping is needed
pipe.connect('eg_writer', 'resolver_title',  {})
pipe.connect('eg_writer', 'resolver_name',   {})
pipe.connect('eg_writer', 'entity_embedder', {})

# 3. Inputs that don't come from upstream components
pipe_inputs = {
    'data_loader': {'path': DATA_PATH},
    'schema': {
        'node_types':         node_types,
        'relationship_types': relationship_types,
        'patterns':           patterns,
    },
}

# 4. Run!
result = await pipe.run(pipe_inputs)
print('\nPipeline finished ✔')
print(result)

## 13. Validation Cypher — check the new movie schema landed correctly

These mirror the validation block in `01_first_nb.ipynb` so you can compare the custom-pipeline output against the `SimpleKGPipeline` baseline.

In [ ]:
import pandas as pd

def run_cypher(query: str, **params) -> pd.DataFrame:
    with driver.session(database=NEO4J_DATABASE) as session:
        res = session.run(query, **params)
        return pd.DataFrame([r.data() for r in res])

In [ ]:
# Node-label distribution
run_cypher('''
MATCH (n)
UNWIND labels(n) AS lbl
RETURN lbl AS label, count(*) AS count
ORDER BY count DESC
''')

In [ ]:
# Relationship-type distribution
run_cypher('''
MATCH ()-[r]->()
RETURN type(r) AS relationshipType, count(*) AS count
ORDER BY count DESC
''')

In [ ]:
# Top actors by number of movies (via CAST_IN)
run_cypher('''
MATCH (p:Person)-[:CAST_IN]->(m:Movie)
RETURN p.name AS actor, count(DISTINCT m) AS movie_count
ORDER BY movie_count DESC
LIMIT 10
''')

In [ ]:
# Top directors by number of movies (via DIRECTED_BY)
run_cypher('''
MATCH (m:Movie)-[:DIRECTED_BY]->(p:Person)
RETURN p.name AS director, count(DISTINCT m) AS movie_count
ORDER BY movie_count DESC
LIMIT 10
''')

In [ ]:
# Movies by genre (via HAS_GENRE)
run_cypher('''
MATCH (m:Movie)-[:HAS_GENRE]->(g:Genre)
RETURN g.name AS genre, count(DISTINCT m) AS movie_count
ORDER BY movie_count DESC
LIMIT 10
''')

In [ ]:
# Sample movie inspection — pull the first Movie and its directly-connected facts
run_cypher('''
MATCH (m:Movie)
WITH m LIMIT 1
OPTIONAL MATCH (m)-[:HAS_GENRE]->(g:Genre)
OPTIONAL MATCH (m)-[:PRODUCED_BY]->(pc:ProductionCompany)
OPTIONAL MATCH (m)-[:PRODUCED_IN]->(c:Country)
OPTIONAL MATCH (m)-[:SPOKEN_IN]->(l:Language)
OPTIONAL MATCH (m)-[:TAGGED_WITH]->(k:Keyword)
OPTIONAL MATCH (m)-[:DIRECTED_BY]->(d:Person)
OPTIONAL MATCH (m)-[:WRITTEN_BY]->(w:Person)
OPTIONAL MATCH (a:Person)-[:CAST_IN]->(m)
RETURN m.title  AS title,
       collect(DISTINCT g.name)  AS genres,
       collect(DISTINCT pc.name) AS production_companies,
       collect(DISTINCT c.name)  AS countries,
       collect(DISTINCT l.name)  AS languages,
       collect(DISTINCT k.name)  AS keywords,
       collect(DISTINCT d.name)  AS directors,
       collect(DISTINCT w.name)  AS writers,
       collect(DISTINCT a.name)[..10] AS sample_actors
''')

In [ ]:
# Confirm entity-level embeddings were written by EntityTextEmbedder
run_cypher('''
MATCH (e:__Entity__)
RETURN count(e) AS total_entities,
       count(e.embedding) AS with_embedding
''')

## 14. Clean shutdown

In [ ]:
driver.close()
print('Driver closed ✔')

Collecting workspace informationFiltering to most relevant information## Analysis

Your **Storyboarding 2.0** concept maps very cleanly onto what you've already built in 03_custom_pipeline_movies_kg.ipynb:

- The **Movie Intelligence Agent (TMD Agent)** = a retriever on top of the Neo4j graph created by your custom pipeline (uses `Movie`, `Person`, `Genre`, `ProductionCompany`, `Keyword`, `CastMember` nodes and `CAST_IN` / `DIRECTED_BY` / `HAS_GENRE` / `TAGGED_WITH` edges).
- The **shared graph memory** = an extension of that same Neo4j Aura database (`NEO4J_URI`) with new node labels for the creative process (`Idea`, `Script`, `Screenplay`, `Scene`, `Storyboard`, `ShortFilm`, `HumanDecision`).
- The **EntityTextEmbedder** custom `Component` you already wrote is exactly the pattern you'd reuse for embedding scripts/scenes/storyboards so agents can semantically retrieve past creative choices.
- The **`Pipeline` + `pipe.connect(...)` wiring** you used in cell `#VSC-4baa2e60 03_custom_pipeline_movies_kg.ipynb` is the same orchestration style you'd use for chaining the 5 creative agents.

Below are three Mermaid diagrams I'd suggest — they render natively inside the notebook (Markdown cell) and in VS Code preview.

---

### Diagram 1 — System Architecture (Human × Agents × Graph Memory)



In [ ]:
```mermaid
flowchart TB
    HUMAN(["🧑 Human Director<br/>(creative lead)"]):::human

    subgraph AGENTS["🤖 Agent Layer"]
        TMD["Movie Intelligence Agent<br/>(TMD / RAG on Movies KG)"]:::agent
        SCRIPT["Scripting Agent"]:::agent
        SCREEN["Screenplay Agent"]:::agent
        STORY["Storyboarding Agent"]:::agent
        FILM["Short Film Agent<br/>(vision / video model)"]:::agent
    end

    subgraph TOOLS["🛠 Tool Layer"]
        WEB["Web Search"]
        IMG["Image Gen"]
        VID["Video Gen"]
    end

    subgraph MEM["🧠 Graph Memory (Neo4j Aura)"]
        direction LR
        KG_MOVIES["Movies KG<br/>Movie · Person · Genre<br/>Keyword · CastMember<br/>(from 03_custom_pipeline_movies_kg.ipynb)"]:::kg
        KG_CREATIVE["Creative KG<br/>Idea · Theme · Script<br/>Screenplay · Scene<br/>Storyboard · ShortFilm<br/>HumanDecision · Feedback"]:::kg
    end

    HUMAN <-->|Stage 1| TMD
    HUMAN <-->|Stage 2| SCRIPT
    HUMAN <-->|Stage 3| SCREEN
    HUMAN <-->|Stage 4 review| STORY
    HUMAN <-->|Stage 5 review| FILM

    SCRIPT  <-->|consults| TMD
    SCREEN  <-->|consults| TMD
    SCRIPT  --> WEB
    STORY   --> IMG
    FILM    --> VID

    TMD     <--> KG_MOVIES
    SCRIPT  <--> KG_CREATIVE
    SCREEN  <--> KG_CREATIVE
    STORY   <--> KG_CREATIVE
    FILM    <--> KG_CREATIVE
    KG_MOVIES -.cross-links<br/>INSPIRED_BY.-> KG_CREATIVE

    classDef human    fill:#FFE9B3,stroke:#C99500,stroke-width:2px,color:#000
    classDef agent    fill:#D9EAFD,stroke:#1F6FEB,color:#000
    classDef kg       fill:#E6F4EA,stroke:#2E7D32,color:#000
```



---

### Diagram 2 — Stage-by-Stage Sequence Flow



In [ ]:
```mermaid
sequenceDiagram
    autonumber
    actor H as 🧑 Human
    participant TMD as Movie Intel Agent
    participant SC as Scripting Agent
    participant SP as Screenplay Agent
    participant SB as Storyboard Agent
    participant SF as Short Film Agent
    participant G as 🧠 Graph Memory

    rect rgb(255,243,205)
    Note over H,TMD: Stage 1 — Idea Discovery
    H->>TMD: 1-liner / genre / question
    TMD->>G: query Movie/Genre/Keyword
    TMD-->>H: reference movies + patterns
    H->>G: MERGE (:Idea {locked:true})
    end

    rect rgb(217,234,253)
    Note over H,SC: Stage 2 — Script
    H->>SC: locked Idea
    SC->>TMD: "what worked for this theme?"
    SC->>G: MERGE (:Script)-[:DERIVED_FROM]->(:Idea)
    SC-->>H: draft + open questions
    H-->>SC: decisions / rejections
    SC->>G: log HumanDecision + RejectedIdea
    end

    rect rgb(230,244,234)
    Note over H,SP: Stage 3 — Screenplay
    SP->>G: read Script + decisions
    SP->>TMD: best screenplays in genre
    SP-->>H: scene breakdown
    H-->>SP: approvals
    SP->>G: MERGE (:Screenplay)-[:CONTAINS_SCENE]->(:Scene)
    end

    rect rgb(252,228,236)
    Note over H,SB: Stage 4 — Storyboard
    SB->>G: read Scenes
    SB-->>H: visual frames per scene
    H-->>SB: approve/revise
    SB->>G: MERGE (:Storyboard {version})
    end

    rect rgb(237,231,246)
    Note over H,SF: Stage 5 — Short Film
    SF->>G: read approved Storyboard
    SF-->>H: short film
    H-->>SF: final feedback
    SF->>G: MERGE (:ShortFilm)-[:APPROVED_BY]->(:Human)
    end
```



---

### Diagram 3 — Extended Graph Schema (your Movies KG + Creative layer)

This shows how to **extend** the schema you defined in the `SchemaBuilder` cell (`#VSC-83432393 03_custom_pipeline_movies_kg.ipynb`) — left side is what already exists, right side is new.



In [ ]:
```mermaid
flowchart LR
    subgraph EXIST["✅ Already in 03_custom_pipeline_movies_kg.ipynb"]
        direction TB
        M[(Movie)]
        P[(Person)]
        G[(Genre)]
        PC[(ProductionCompany)]
        K[(Keyword)]
        CM[(CastMember)]
        C[(Country)]
        L[(Language)]

        M -- HAS_GENRE   --> G
        M -- TAGGED_WITH --> K
        M -- DIRECTED_BY --> P
        M -- WRITTEN_BY  --> P
        P  -- CAST_IN     --> M
        M -- PRODUCED_BY --> PC
        M -- PRODUCED_IN --> C
        M -- SPOKEN_IN   --> L
    end

    subgraph NEW["🆕 Creative layer to add"]
        direction TB
        ID[(Idea)]
        TH[(Theme)]
        SCR[(Script)]
        SCP[(Screenplay)]
        SCN[(Scene)]
        CH[(Character)]
        SB[(Storyboard)]
        SF[(ShortFilm)]
        HD[(HumanDecision)]
        FB[(Feedback)]
        RJ[(RejectedIdea)]

        ID -- HAS_THEME       --> TH
        ID -- DERIVED_FROM    --> SCR
        SCR -- BECOMES        --> SCP
        SCP -- CONTAINS_SCENE --> SCN
        SCN -- HAS_STORYBOARD --> SB
        SB  -- RENDERED_AS    --> SF
        SCN -- FEATURES       --> CH
        HD  -- APPLIES_TO     --> ID
        HD  -- APPLIES_TO     --> SCR
        FB  -- ON             --> SB
        RJ  -- SUPERSEDED_BY  --> ID
    end

    %% Cross-links between the two subgraphs (the magic)
    ID  -. INSPIRED_BY .-> M
    SCR -. REFERENCES  .-> M
    SCP -. STYLE_OF    .-> P
    CH  -. ARCHETYPE_OF .-> CM
    TH  -. ALIGNS_WITH .-> G
```



---

## What you can build with this (concrete next steps)

| Step | Reuse from your notebook | New work |
|---|---|---|
| Movie Intel Agent | Existing Neo4j KG + `EntityTextEmbedder` embeddings | Wrap in a LangChain/`neo4j_graphrag` retriever tool |
| Creative KG schema | Extend the `SchemaBuilder` cell `#VSC-83432393 03_custom_pipeline_movies_kg.ipynb` with the new `NodeType`s above | Add `Idea`, `Script`, `Scene`, `Storyboard`, `ShortFilm`, `HumanDecision` |
| Pipeline wiring | Reuse the `Pipeline().add_component()` + `pipe.connect(...)` pattern from cell `#VSC-4baa2e60 03_custom_pipeline_movies_kg.ipynb` | One pipeline **per stage** (5 pipelines), chained via the graph |
| Memory writes | Reuse `Neo4jWriter` with the same `driver` + `NEO4J_DATABASE` | Add a `HumanDecisionWriter` custom `Component` (same pattern as `EntityTextEmbedder`) |
| Embeddings on creative nodes | Reuse `EntityTextEmbedder` — it already embeds the **longest string property** of any `:__Entity__` node | Just label `Script`/`Scene`/`Storyboard` with `:__Entity__` so it auto-embeds |
| Human-in-the-loop | — | Chainlit UI (you already have Inference Pipeline/Chainlit_App) wired to the 5 agents |

The big insight: because your existing pipeline already produces an `:__Entity__`-labelled graph with embeddings, **every new creative node automatically becomes semantically searchable** — which is exactly what "living graph memory" needs.